In [1]:
!pip install tabulate

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, matthews_corrcoef, classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from textwrap import shorten
import warnings
from sklearn.exceptions import ConvergenceWarning   # ← add this

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# 1) Load & balance
df = pd.read_csv('ai4i2020.csv')
feature_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]
X = df[feature_cols]
y = df["Machine failure"]

# try imblearn, fallback to pandas sampling
try:
    from imblearn.under_sampling import RandomUnderSampler
    rus = RandomUnderSampler(sampling_strategy={0:339, 1:339}, random_state=42)
    X_bal, y_bal = rus.fit_resample(X, y)
except ImportError:
    df_min = df[df['Machine failure']==1].sample(339, random_state=42)
    df_maj = df[df['Machine failure']==0].sample(339, random_state=42)
    df_bal = pd.concat([df_min, df_maj]).sample(frac=1, random_state=42)
    X_bal, y_bal = df_bal[feature_cols], df_bal["Machine failure"]

# 5. Verify balanced counts
print("Balanced class counts:")
print(y_bal.value_counts(), "\n")


# 2) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)

# quick sanity‑check of the split
n_total   = len(X_bal)
n_train   = len(X_train)
n_test    = len(X_test)

print(f"Total rows : {n_total}")
print(f"Train rows : {n_train}  ({n_train / n_total:.2%})")
print(f"Test  rows : {n_test}   ({n_test  / n_total:.2%})")

# verify class distribution stayed the same
print("\nClass distribution (train):")
print(y_train.value_counts(normalize=True))
print("\nClass distribution (test):")
print(y_test.value_counts(normalize=True))

# optional hard assertions (raises AssertionError if not ~80/20)
print(f"Train %: {n_train/n_total:.2%} | Test %: {n_test/n_total:.2%}")

# 3) Preprocessing
preprocessor = ColumnTransformer([
    ("scale", StandardScaler(), feature_cols)
])
mcc_scorer = make_scorer(matthews_corrcoef)

# 4) Search configs
search_configs = {
    "MLP": {
        "type": "random", "n_iter": 20,
        "pipeline": Pipeline([("scale", preprocessor), 
                              ("clf", MLPClassifier(max_iter=500, random_state=42))]),
        "params": {
            "clf__hidden_layer_sizes": [(50,),(100,),(50,50),(100,50)],
            "clf__activation": ["relu","tanh","logistic"],
            "clf__learning_rate": ["constant","adaptive"],
            "clf__alpha": [1e-5,1e-4,1e-3]
        }
    },
    "SVM": {
        "type": "random", "n_iter": 20,
        "pipeline": Pipeline([("scale", preprocessor), ("clf", SVC(random_state=42))]),
        "params": {
            "clf__C": [0.1,1,10,100],
            "clf__kernel": ["linear","rbf","poly"],
            "clf__gamma": ["scale","auto"]
        }
    },
    "KNN": {
        "type": "random", "n_iter": 15,
        "pipeline": Pipeline([("scale", preprocessor), ("clf", KNeighborsClassifier())]),
        "params": {
            "clf__n_neighbors": [3,5,7,9,11],
            "clf__p": [1,2],
            "clf__algorithm": ["auto","ball_tree","kd_tree"]
        }
    },
    "DecisionTree": {
        "type": "grid",
        "pipeline": Pipeline([("scale", preprocessor), ("clf", DecisionTreeClassifier(random_state=42))]),
        "params": {
            "clf__criterion": ["gini","entropy"],
            "clf__max_depth": [None,5,10,20],
            "clf__ccp_alpha": [0.0,0.01,0.05]
        }
    },
    "LogisticRegression": {
        "type": "grid",
        "pipeline": Pipeline([("scale", preprocessor), ("clf", LogisticRegression(solver="liblinear", random_state=42))]),
        "params": {
            "clf__penalty": ["l2","l1"],
            "clf__C": [0.01,0.1,1,10,100],
            "clf__solver": ["liblinear"]
        }
    }
}



import pandas as pd
from textwrap import shorten
from sklearn.metrics import matthews_corrcoef

pd.set_option("display.width", 0)          # let tables go >80 cols
pd.set_option("display.max_colwidth", None)

# ───────────────────────────────────────────────────────────────
# 5) Hyper‑parameter search  →  TABLE 1 (80 % train, 5‑fold CV)
# ───────────────────────────────────────────────────────────────
best_estimators, param_map, rows_cv = {}, {}, []

for name, cfg in search_configs.items():
    # choose silent searcher
    if cfg["type"] == "random":
        searcher = RandomizedSearchCV(
            cfg["pipeline"], cfg["params"],
            n_iter=cfg["n_iter"], scoring=mcc_scorer,
            cv=5, random_state=42, n_jobs=-1, verbose=0
        )
    else:
        searcher = GridSearchCV(
            cfg["pipeline"], cfg["params"],
            scoring=mcc_scorer, cv=5, n_jobs=-1, verbose=0
        )

    searcher.fit(X_train, y_train)

    best_estimators[name] = searcher.best_estimator_
    param_str_full = str(searcher.best_params_)           # full, un‑truncated
    param_map[name] = param_str_full                      # ← used by Table 2

    rows_cv.append({
        "ML Trained Model": name,
        "Its Best Set of Parameter Values": param_str_full,
        "MCC on 5‑fold CV\n(80 % train)": f"{searcher.best_score_:.4f}"
    })

table1 = pd.DataFrame(rows_cv)
print("\n### TABLE 1 – Training‑CV ###")
print(table1.to_markdown(index=False))

# ───────────────────────────────────────────────────────────────
# 6) Test‑set evaluation  →  TABLE 2 (20 % hold‑out)
# ───────────────────────────────────────────────────────────────
rows_test = []

for name, est in best_estimators.items():
    y_pred = est.predict(X_test)
    mcc    = matthews_corrcoef(y_test, y_pred)

    rows_test.append({
        "ML Trained Model": name,
        "Its Best Set of Parameter Values": param_map[name],  # reuse string
        "MCC on 20 % Test set": f"{mcc:.4f}"
    })

table2 = pd.DataFrame(rows_test)
print("\n### TABLE 2 – Hold‑out Test ###")
print(table2.to_markdown(index=False))





Balanced class counts:
Machine failure
0    339
1    339
Name: count, dtype: int64 

Total rows : 678
Train rows : 542  (79.94%)
Test  rows : 136   (20.06%)

Class distribution (train):
Machine failure
1    0.5
0    0.5
Name: proportion, dtype: float64

Class distribution (test):
Machine failure
1    0.5
0    0.5
Name: proportion, dtype: float64
Train %: 79.94% | Test %: 20.06%


/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't conver


### TABLE 1 – Training‑CV ###
| ML Trained Model   | Its Best Set of Parameter Values                                                                                         |   MCC on 5‑fold CV |
|                    |                                                                                                                          |       (80 % train) |
|:-------------------|:-------------------------------------------------------------------------------------------------------------------------|-------------------:|
| MLP                | {'clf__learning_rate': 'constant', 'clf__hidden_layer_sizes': (100, 50), 'clf__alpha': 0.001, 'clf__activation': 'relu'} |             0.8232 |
| SVM                | {'clf__kernel': 'rbf', 'clf__gamma': 'auto', 'clf__C': 100}                                                              |             0.8196 |
| KNN                | {'clf__p': 2, 'clf__n_neighbors': 7, 'clf__algorithm': 'ball_tree'}                                            

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
X_bal